# IndexTTS2 - 部署笔记本 (Colab/Kaggle)\n\n**部署流程**: ①安装mamba → ②创建Python 3.10环境 → ③安装依赖 → ④克隆代码

In [ ]:
# ============================================# Cell 1: 环境准备 (3步流程)# ============================================import os, sys, json, subprocess, urllib.request# 基础配置IN_COLAB = 'google.colab' in sys.modulesIN_KAGGLE = os.path.exists('/kaggle/input')WORK_DIR = '/content' if IN_COLAB else '/kaggle/working' if IN_KAGGLE else '/tmp'REPO_DIR = f"{WORK_DIR}/index-tts"MAMBA_ROOT = f"{WORK_DIR}/mamba"MAMBA_ENV = f"{MAMBA_ROOT}/envs/tts"print(f"🔍 环境: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Other'}")print(f"📁 工作目录: {WORK_DIR}")print("\n" + "="*60)# =====================================================# 第1步: 安装mamba (必须成功)# =====================================================print("\n📦 第1步: 安装mamba...")MICROMAMBA_BIN = None# 1.1 尝试使用现有的mamba (必须能执行create命令)def check_existing_mamba():    """检查系统中是否有真正的mamba包管理器 (不是测试框架)"""    try:        result = subprocess.run(['which', 'mamba'], capture_output=True, text=True)        if result.returncode == 0:            bin_path = result.stdout.strip()            # 关键测试：执行 create --help 看是否支持包管理功能            test_result = subprocess.run([bin_path, 'create', '--help'],                                         capture_output=True, text=True, timeout=5)            # 真正的mamba会显示create命令的帮助，测试框架会报错            if test_result.returncode == 0 and 'environment' in test_result.stdout.lower():                print(f"✅ 发现真正的mamba包管理器: {bin_path}")                return bin_path            else:                print(f"⚠️ {bin_path} 不是包管理器 (可能是测试框架)，跳过")    except Exception as e:        pass    return NoneMICROMAMBA_BIN = check_existing_mamba()# 1.2 如果没有，安装micromambaif not MICROMAMBA_BIN:    print("🔧 安装 micromamba...")        os.environ['MAMBA_ROOT_PREFIX'] = MAMBA_ROOT        install_script = f"{WORK_DIR}/install.sh"    urllib.request.urlretrieve('https://micro.mamba.pm/install.sh', install_script)    os.chmod(install_script, 0o755)        subprocess.run(['bash', install_script], check=True)        # 查找micromamba    for path in [        '/root/.local/bin/micromamba',        f"{os.environ.get('HOME', '/root')}/.local/bin/micromamba",        f"{MAMBA_ROOT}/bin/micromamba"    ]:        if os.path.exists(path):            MICROMAMBA_BIN = path            print(f"✅ 安装成功: {path}")            break        if not MICROMAMBA_BIN:        raise RuntimeError("❌ mamba安装失败，无法继续")# 1.3 验证并添加到PATHbin_dir = os.path.dirname(MICROMAMBA_BIN)os.environ['PATH'] = f"{bin_dir}:{os.environ['PATH']}"# 创建mamba别名mamba_link = os.path.join(bin_dir, 'mamba')if not os.path.exists(mamba_link):    os.symlink(MICROMAMBA_BIN, mamba_link)print(f"✅ mamba就绪: {MICROMAMBA_BIN}")# =====================================================# 第2步: 创建Python 3.10环境 (必须成功)# =====================================================print("\n📦 第2步: 创建Python 3.10环境...")print("⏳ 安装python=3.10和cudatoolkit=11.8，请等待...")# 尝试创建带CUDA的环境cmd = f"{MICROMAMBA_BIN} create -y -p {MAMBA_ENV} python=3.10 cudatoolkit=11.8 -c conda-forge -c nvidia"result = subprocess.run(cmd, shell=True, capture_output=True, text=True)if result.returncode != 0:    print("⚠️ 带CUDA的环境创建失败，尝试仅安装Python...")    # Fallback: 只安装python    cmd = f"{MICROMAMBA_BIN} create -y -p {MAMBA_ENV} python=3.10 -c conda-forge"    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)        if result.returncode != 0:        print(f"❌ 环境创建失败:")        print(result.stderr)        raise RuntimeError("环境创建失败")    else:        print("✅ Python 3.10环境创建成功 (无CUDA)")print("✅ Python 3.10环境创建成功")# 设置Python和pip路径MAMBA_PYTHON = f"{MAMBA_ENV}/bin/python"MAMBA_PIP = f"{MAMBA_ENV}/bin/pip"print(f"   Python: {MAMBA_PYTHON}")print(f"   pip: {MAMBA_PIP}")# =====================================================# 第3步: 安装依赖 (使用mamba环境的pip)# =====================================================print("\n📦 第3步: 安装Python依赖...")# 3.1 PyTorch (项目要求版本)print("   安装 PyTorch (尝试cu128)...")try:    # 首先尝试项目要求的cu128    subprocess.run([        MAMBA_PIP, 'install', '-q',        'torch==2.8.0', 'torchaudio==2.8.0',        '--index-url', 'https://download.pytorch.org/whl/cu128'    ], check=True)    print("   ✅ PyTorch cu128 安装成功")except:    print("   ⚠️ cu128不可用，尝试cu121...")    try:        # 回退到cu121        subprocess.run([            MAMBA_PIP, 'install', '-q',            'torch==2.5.1', 'torchaudio==2.5.1',            '--index-url', 'https://download.pytorch.org/whl/cu121'        ], check=True)        print("   ✅ PyTorch cu121 (2.5.1) 安装成功")    except:        print("   ⚠️ CUDA版本均不可用，使用标准pip安装...")        # 最后回退到标准pip（无CUDA指定）        subprocess.run([MAMBA_PIP, 'install', '-q', 'torch', 'torchaudio'], check=True)        print("   ✅ PyTorch (标准版) 安装成功")# 3.2 基础依赖print("   安装基础依赖...")deps = ["pyngrok", "gradio==5.45.0", "fastapi", "uvicorn", "python-multipart", "huggingface-hub", "git-lfs", "requests"]for dep in deps:    subprocess.run([MAMBA_PIP, 'install', '-q', dep], check=True)print("✅ 依赖安装完成")# =====================================================# 第4步: 克隆代码# =====================================================print("\n📦 第4步: 克隆代码...")if os.path.exists(REPO_DIR):    subprocess.run(['rm', '-rf', REPO_DIR], check=True)# 跳过LFS文件env = os.environ.copy()env['GIT_LFS_SKIP_SMUDGE'] = '1'subprocess.run([    'git', 'clone', '--depth', '1', '-b', 'dev',    'https://github.com/infinite-gaming-studio/index-tts.git',    REPO_DIR], env=env, check=True)print(f"✅ 代码克隆完成: {REPO_DIR}")# =====================================================# 保存配置# =====================================================config = {    'work_dir': WORK_DIR,    'repo_dir': REPO_DIR,    'mamba_env': MAMBA_ENV,    'python': MAMBA_PYTHON,    'pip': MAMBA_PIP,    'in_colab': IN_COLAB,    'in_kaggle': IN_KAGGLE}with open('/tmp/notebook_config.json', 'w') as f:    json.dump(config, f, indent=2)print("\n" + "="*60)print("✅ Cell 1 完成!")print(f"   Python: {MAMBA_PYTHON}")print(f"   代码: {REPO_DIR}")print(f"\n👉 请重新启动kernel，然后运行Cell 2")print(f"   (使用: {MAMBA_PYTHON})")print("="*60)

In [ ]:
# ============================================# Cell 2: 安装项目依赖和模型# 确保使用mamba环境的Python运行此Cell# ============================================import sys, json, subprocess, os# 加载配置with open('/tmp/notebook_config.json', 'r') as f:    config = json.load(f)REPO_DIR = config['repo_dir']PYTHON = config['python']PIP = config['pip']print(f"✅ 使用mamba Python: {PYTHON}")# 添加项目路径sys.path.insert(0, REPO_DIR)# 安装项目依赖print("\n📦 安装项目依赖...")from deploy.utils import DependencyInstaller, ModelDownloader, check_model_existsDependencyInstaller.install_deps(use_mamba=False)# 安装项目print("\n📦 安装项目...")subprocess.run([PIP, 'install', '-q', '-e', REPO_DIR], check=True)# 下载模型print("\n🤖 检查模型...")CHECKPOINT_DIR = f"{REPO_DIR}/checkpoints"# 统一检查逻辑：config.yaml + 模型权重文件def check_model_complete(checkpoint_dir):    """检查模型是否完整（配置文件+权重文件）"""    config_file = os.path.join(checkpoint_dir, "config.yaml")    if not os.path.exists(config_file):        return False, "配置文件不存在"        if not os.path.isdir(checkpoint_dir):        return False, "检查点目录不存在"        files = os.listdir(checkpoint_dir)    model_files = [f for f in files if f.endswith(('.pt', '.bin', '.pth', '.safetensors', '.ckpt'))]        if not model_files:        return False, f"无权重文件，现有文件: {files}"        return True, f"找到 {len(model_files)} 个权重文件"is_complete, msg = check_model_complete(CHECKPOINT_DIR)print(f"   检查结果: {msg}")if not is_complete:    print("   模型不完整，开始下载...")    ModelDownloader.download(CHECKPOINT_DIR)    # 下载后再次检查    is_complete, msg = check_model_complete(CHECKPOINT_DIR)    print(f"   下载后检查: {msg}")else:    print(f"   ✅ 模型已完整存在")print("\n" + "="*60)print("✅ Cell 2 完成! 可以启动服务了")print("="*60)

In [ ]:
# ============================================# Cell 3: 启动服务 (实时日志输出 + 就绪检测)# ============================================import jsonimport subprocessimport osimport sysimport timeimport threadingimport requests# 加载配置with open('/tmp/notebook_config.json', 'r') as f:    config = json.load(f)REPO_DIR = config['repo_dir']PYTHON = config['python']PORT = 8000MODE = "both"NGROK_TOKEN = None  # 如需ngrok，填入token: "your_token_here"print(f"🚀 准备启动服务...")print(f"   Python: {PYTHON}")print(f"   代码: {REPO_DIR}")print(f"   端口: {PORT}")# 检查模型是否完整（与Cell 2使用相同逻辑）checkpoint_dir = os.path.join(REPO_DIR, "checkpoints")config_file = os.path.join(checkpoint_dir, "config.yaml")# 显示目录内容用于诊断print(f"\n🔍 检查模型目录: {checkpoint_dir}")if os.path.exists(checkpoint_dir):    all_files = os.listdir(checkpoint_dir)    print(f"   目录存在，文件数: {len(all_files)}")    if all_files:        print(f"   文件列表: {all_files[:10]}{'...' if len(all_files) > 10 else ''}")    model_files = [f for f in all_files if f.endswith(('.pt', '.bin', '.pth', '.safetensors', '.ckpt'))]    print(f"   模型权重文件: {len(model_files)}个")else:    print(f"   ❌ 目录不存在")    all_files = []    model_files = []# 统一检查逻辑config_exists = os.path.exists(config_file)has_weights = len(model_files) > 0if not config_exists:    print(f"\n❌ 错误: 模型配置文件不存在")    print(f"   路径: {config_file}")    print(f"\n请先运行 Cell 2 下载模型")elif not has_weights:    print(f"\n❌ 错误: 模型权重文件不存在")    print(f"   检查点目录: {checkpoint_dir}")    print(f"   现有文件: {all_files}")    print(f"\n⚠️ 模型下载不完整，请重新运行 Cell 2")else:    print(f"\n✅ 模型检查通过 (配置文件 + {len(model_files)}个权重文件)")# 仅在模型完整时启动if config_exists and has_weights:    # 检查系统资源    print(f"\n📊 系统资源检查:")    try:        import torch        if torch.cuda.is_available():            gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9            print(f"   GPU: {torch.cuda.get_device_name(0)}")            print(f"   GPU显存: {gpu_mem:.1f} GB")        else:            print(f"   ⚠️ 警告: 没有可用的GPU，模型加载可能很慢或失败")    except Exception as e:        print(f"   无法检查GPU: {e}")        print(f"\n🚀 启动服务...")    print(f"   ⚠️ 模型加载需要1-3分钟，请耐心等待...")    print(f"\n{'='*60}")    print("📋 服务日志输出:")    print(f"{'='*60}\n")        # 直接启动 service.py，避免 launcher 的中间层    service_script = os.path.join(REPO_DIR, "deploy", "service.py")        # 设置环境变量    env = os.environ.copy()    env["INDEXTTS_REPO_DIR"] = REPO_DIR    env["PYTHONUNBUFFERED"] = "1"  # 确保Python输出无缓冲        # 启动进程，使用管道捕获输出    proc = subprocess.Popen(        [PYTHON, '-u', service_script, "--port", str(PORT), "--mode", MODE],        stdout=subprocess.PIPE,        stderr=subprocess.STDOUT,        cwd=REPO_DIR,        env=env,        bufsize=1,        universal_newlines=True    )        # 用于存储输出和状态的变量    output_lines = []    service_ready = threading.Event()    model_loaded = threading.Event()    stop_output_thread = threading.Event()        def output_reader():        """实时读取并打印子进程输出"""        for line in proc.stdout:            line = line.rstrip()            output_lines.append(line)            print(line, flush=True)  # 实时输出                        # 检测模型加载完成的关键字            if "模型加载完成" in line or "模型加载完成" in line or "✅ 模型加载完成" in line:                model_loaded.set()                        if len(output_lines) > 1000:  # 限制内存使用                output_lines.pop(0)                # 进程结束        if not stop_output_thread.is_set():            stop_output_thread.set()        # 启动输出读取线程    output_thread = threading.Thread(target=output_reader, daemon=True)    output_thread.start()        # 等待服务初始化（最多等待5分钟）    max_wait_time = 300  # 5分钟    check_interval = 2   # 每2秒检查一次    elapsed = 0        print(f"\n⏳ 等待服务就绪...")        while elapsed < max_wait_time:        # 检查进程是否还在运行        if proc.poll() is not None:            return_code = proc.returncode            print(f"\n{'='*60}")            print(f"❌ 服务启动失败 (返回码: {return_code})")            if return_code == -15:                print("\n⚠️ 进程被信号15(SIGTERM)终止，可能原因:")                print("   1. 内存不足 (Kaggle/Colab有内存限制)")                print("   2. 模型加载超时")                print("   3. 系统资源限制")                print("\n💡 建议:")                print("   - 检查GPU内存是否足够")                print("   - 尝试使用更小的模型或启用CPU模式")                print("   - 在Kaggle设置中启用GPU加速")            print(f"{'='*60}")            break                # 尝试连接健康检查端点        try:            response = requests.get(f"http://localhost:{PORT}/api/health", timeout=2)            if response.status_code == 200:                health_data = response.json()                if health_data.get("loaded"):                    service_ready.set()                    print(f"\n{'='*60}")                    print(f"✅ 服务完全就绪!")                    print(f"   模型设备: {health_data.get('device', 'unknown')}")                    print(f"   加载状态: {'已加载' if health_data.get('loaded') else '未加载'}")                    break                else:                    print(f"   服务运行中但模型尚未加载完成... ({elapsed}s)")        except requests.exceptions.ConnectionError:            if elapsed % 10 == 0:  # 每10秒打印一次                print(f"   等待服务启动... ({elapsed}s)")        except requests.exceptions.Timeout:            pass        except Exception as e:            if elapsed % 10 == 0:                print(f"   等待服务启动... ({elapsed}s)")                time.sleep(check_interval)        elapsed += check_interval        if service_ready.is_set():        print(f"{'='*60}\n")        print(f"🔗 本地地址: http://localhost:{PORT}")        print(f"📖 API文档: http://localhost:{PORT}/docs")        print(f"🖥️  WebUI: http://localhost:{PORT}/ui")        print(f"\n{'='*60}")        print("✅ 服务已成功启动!")        print("="*60)                # 继续等待进程（保持服务运行）        try:            while proc.poll() is None:                time.sleep(1)        except KeyboardInterrupt:            print("\n🛑 收到中断信号，停止服务...")            stop_output_thread.set()            proc.terminate()            proc.wait(timeout=10)            print("✅ 服务已停止")                elif elapsed >= max_wait_time:        print(f"\n{'='*60}")        print("⚠️ 等待超时 (5分钟)")        print("   服务可能仍在加载中，但已超过最大等待时间")        print("\n💡 可能原因:")        print("   1. 模型文件损坏或不完整")        print("   2. GPU内存不足")        print("   3. 网络问题导致下载不完整")        print("\n💡 建议操作:")        print("   - 检查上面的日志输出是否有错误信息")        print("   - 重新运行 Cell 2 下载模型")        print("   - 检查GPU内存是否足够加载模型")        print(f"{'='*60}")                # 终止进程        proc.terminate()        proc.wait(timeout=10)        else:    print(f"\n⚠️ 跳过服务启动 (模型未就绪)")

In [ ]:
# ============================================# Cell 4: 检查服务状态# ============================================import jsonimport osimport subprocessimport requestswith open('/tmp/notebook_config.json', 'r') as f:    config = json.load(f)REPO_DIR = config['repo_dir']PORT = 8000print("🔍 检查服务状态...")print("="*60)# 检查进程result = subprocess.run(['ps', 'aux'], capture_output=True, text=True)if 'launcher.py' in result.stdout or 'service.py' in result.stdout:    print("✅ 服务进程正在运行")else:    print("❌ 服务进程未运行")# 检查端口result = subprocess.run(['netstat', '-tlnp'], capture_output=True, text=True)if f':{PORT}' in result.stdout:    print(f"✅ 端口 {PORT} 正在监听")else:    print(f"⚠️ 端口 {PORT} 未监听")# 检查API健康状态print("\n📊 API健康检查:")try:    response = requests.get(f"http://localhost:{PORT}/api/health", timeout=5)    if response.status_code == 200:        health = response.json()        print(f"   状态: {health.get('status', 'unknown')}")        print(f"   模型: {health.get('model', 'unknown')}")        print(f"   设备: {health.get('device', 'unknown')}")        print(f"   已加载: {'✅ 是' if health.get('loaded') else '❌ 否'}")    else:        print(f"   ⚠️ API返回状态码: {response.status_code}")except requests.exceptions.ConnectionError:    print("   ❌ 无法连接到服务")except Exception as e:    print(f"   ⚠️ 检查失败: {e}")print("="*60)